In [1]:
# ============================================================
# PHASE 22: HYBRID / ENSEMBLE MODEL COMPARISON
# RTX 5050 - 8GB VRAM OPTIMIZED
# ============================================================

import gc
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 70)
print("PHASE 22 - HYBRID MODEL EXPERIMENT")
print("=" * 70)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

NUM_CLASSES = 7
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4
PIN_MEMORY = True if torch.cuda.is_available() else False

PHASE 22 - HYBRID MODEL EXPERIMENT
Device: cuda
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
GPU Memory: 7.53 GB


In [2]:
# ============================================================
# PATH CONFIGURATION
# ============================================================

BASE_DIR = Path("..")

DATASET_DIR = BASE_DIR / "dataset"
VAL_DIR = DATASET_DIR / "val"

MODELS_DIR = BASE_DIR / "models"

RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Validation folder:", VAL_DIR)
print("Models folder:", MODELS_DIR)
print("Results folder:", RESULTS_DIR)

assert VAL_DIR.exists(), f"Validation folder not found: {VAL_DIR}"
assert MODELS_DIR.exists(), f"Models folder not found: {MODELS_DIR}"

Validation folder: ../dataset/val
Models folder: ../models
Results folder: ../results


In [3]:
# ============================================================
# VALIDATION TRANSFORM
# IMPORTANT: NO AUGMENTATION
# ============================================================

val_transform = transforms.Compose([
    
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    
    transforms.ToTensor(),
    
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# VALIDATION DATASET
# ============================================================

val_dataset = datasets.ImageFolder(
    root=VAL_DIR,
    transform=val_transform
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)


print("=" * 70)
print("VALIDATION DATASET")
print("=" * 70)

print("Number of validation images:", len(val_dataset))
print("Classes:", val_dataset.classes)

print("\nClass mapping:")
print(val_dataset.class_to_idx)

assert len(val_dataset) == 1998, \
    f"Expected 1998 validation images, found {len(val_dataset)}"

assert len(val_dataset.classes) == 7

print("\nVALIDATION DATASET VERIFIED SUCCESSFULLY")

VALIDATION DATASET
Number of validation images: 1998
Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

Class mapping:
{'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}

VALIDATION DATASET VERIFIED SUCCESSFULLY


In [4]:
# ============================================================
# MODEL CREATION FUNCTIONS
# ============================================================


def create_densenet121():
    
    model = models.densenet121(weights=None)
    
    num_features = model.classifier.in_features
    
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(num_features, NUM_CLASSES)
    )
    
    return model


def create_resnet50():
    
    model = models.resnet50(weights=None)
    
    num_features = model.fc.in_features
    
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(num_features, NUM_CLASSES)
    )
    
    return model


def create_efficientnet_b0():
    
    model = models.efficientnet_b0(weights=None)
    
    num_features = model.classifier[1].in_features
    
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(num_features, NUM_CLASSES)
    )
    
    return model


def create_mobilenet_v3_large():
    
    model = models.mobilenet_v3_large(weights=None)
    
    num_features = model.classifier[3].in_features
    
    model.classifier[3] = nn.Linear(
        num_features,
        NUM_CLASSES
    )
    
    return model


def create_swin_tiny():
    
    model = models.swin_t(weights=None)
    
    num_features = model.head.in_features
    
    model.head = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(num_features, NUM_CLASSES)
    )
    
    return model


MODEL_BUILDERS = {
    
    "DenseNet121": create_densenet121,
    
    "ResNet50": create_resnet50,
    
    "EfficientNet_B0": create_efficientnet_b0,
    
    "MobileNetV3_Large": create_mobilenet_v3_large,
    
    "Swin_Tiny": create_swin_tiny
}


print("Model architectures ready:")
for name in MODEL_BUILDERS:
    print("✓", name)

Model architectures ready:
✓ DenseNet121
✓ ResNet50
✓ EfficientNet_B0
✓ MobileNetV3_Large
✓ Swin_Tiny


In [5]:
# ============================================================
# MODEL CHECKPOINT PATHS
# ============================================================

MODEL_PATHS = {
    
    "DenseNet121":
        MODELS_DIR / "densenet121_baseline_best.pth",
    
    
    "ResNet50":
        MODELS_DIR / "resnet50_best.pth",
    
    
    "EfficientNet_B0":
        MODELS_DIR / "efficientnet_b0_best.pth",
    
    
    "MobileNetV3_Large":
        MODELS_DIR / "mobilenetv3_large_best.pth",
    
    
    "Swin_Tiny":
        MODELS_DIR / "swin_tiny_best.pth"
}


print("=" * 70)
print("CHECKING MODEL FILES")
print("=" * 70)

missing = []

for name, path in MODEL_PATHS.items():
    
    if path.exists():
        print(f"✓ {name}: {path}")
    else:
        print(f"✗ {name}: NOT FOUND -> {path}")
        missing.append(name)


assert len(missing) == 0, \
    f"Missing model checkpoints: {missing}"

print("\nALL MODEL FILES FOUND SUCCESSFULLY")

CHECKING MODEL FILES
✓ DenseNet121: ../models/densenet121_baseline_best.pth
✓ ResNet50: ../models/resnet50_best.pth
✓ EfficientNet_B0: ../models/efficientnet_b0_best.pth
✓ MobileNetV3_Large: ../models/mobilenetv3_large_best.pth
✓ Swin_Tiny: ../models/swin_tiny_best.pth

ALL MODEL FILES FOUND SUCCESSFULLY


In [6]:
# ============================================================
# LOAD MODEL
# ============================================================

def load_model(model_name, model_path):
    
    print(f"\nLoading {model_name}...")
    
    # Create architecture
    model = MODEL_BUILDERS[model_name]()
    
    
    # Load checkpoint
    checkpoint = torch.load(
        model_path,
        map_location=device
    )
    
    
    # Some checkpoints save state_dict directly
    if isinstance(checkpoint, dict):
        
        if "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        
        elif "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        
        else:
            state_dict = checkpoint
    
    else:
        state_dict = checkpoint
    
    
    model.load_state_dict(state_dict)
    
    model = model.to(device)
    
    model.eval()
    
    print(f"✓ {model_name} loaded successfully")
    
    return model

In [7]:
# ============================================================
# GENERATE MODEL PROBABILITIES
# ============================================================

def get_model_probabilities(model, loader):
    
    all_probabilities = []
    all_labels = []
    
    model.eval()
    
    with torch.no_grad():
        
        for images, labels in loader:
            
            images = images.to(
                device,
                non_blocking=True
            )
            
            outputs = model(images)
            
            probabilities = torch.softmax(
                outputs,
                dim=1
            )
            
            # Move immediately to CPU
            all_probabilities.append(
                probabilities.cpu()
            )
            
            all_labels.append(
                labels.cpu()
            )
    
    
    all_probabilities = torch.cat(
        all_probabilities,
        dim=0
    ).numpy()
    
    
    all_labels = torch.cat(
        all_labels,
        dim=0
    ).numpy()
    
    
    return all_probabilities, all_labels

In [8]:
# ============================================================
# GENERATE PROBABILITIES FOR ALL MODELS
# ONE MODEL AT A TIME
# ============================================================

all_probabilities = {}

true_labels = None


for model_name, model_path in MODEL_PATHS.items():
    
    print("\n" + "=" * 70)
    print(f"PROCESSING: {model_name}")
    print("=" * 70)
    
    
    # --------------------------------------------------------
    # 1. Load only ONE model
    # --------------------------------------------------------
    
    model = load_model(
        model_name,
        model_path
    )
    
    
    # --------------------------------------------------------
    # 2. Generate validation probabilities
    # --------------------------------------------------------
    
    probabilities, labels = get_model_probabilities(
        model,
        val_loader
    )
    
    
    all_probabilities[model_name] = probabilities
    
    
    # --------------------------------------------------------
    # 3. Verify labels
    # --------------------------------------------------------
    
    if true_labels is None:
        
        true_labels = labels
    
    else:
        
        assert np.array_equal(
            true_labels,
            labels
        ), "Label order mismatch!"
    
    
    print(
        "Probability shape:",
        probabilities.shape
    )
    
    
    # --------------------------------------------------------
    # 4. DELETE MODEL FROM GPU MEMORY
    # --------------------------------------------------------
    
    del model
    
    gc.collect()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    
    print(
        f"✓ {model_name} completed and removed from GPU"
    )
    
    
    if torch.cuda.is_available():
        
        allocated = torch.cuda.memory_allocated() / 1024**2
        
        print(
            f"Current GPU allocated memory: "
            f"{allocated:.2f} MB"
        )


print("\n" + "=" * 70)
print("ALL MODEL PROBABILITIES GENERATED SUCCESSFULLY")
print("=" * 70)

for name, probs in all_probabilities.items():
    print(name, "->", probs.shape)


PROCESSING: DenseNet121

Loading DenseNet121...
✓ DenseNet121 loaded successfully
Probability shape: (1998, 7)
✓ DenseNet121 completed and removed from GPU
Current GPU allocated memory: 32.00 MB

PROCESSING: ResNet50

Loading ResNet50...
✓ ResNet50 loaded successfully
Probability shape: (1998, 7)
✓ ResNet50 completed and removed from GPU
Current GPU allocated memory: 32.00 MB

PROCESSING: EfficientNet_B0

Loading EfficientNet_B0...
✓ EfficientNet_B0 loaded successfully
Probability shape: (1998, 7)
✓ EfficientNet_B0 completed and removed from GPU
Current GPU allocated memory: 32.00 MB

PROCESSING: MobileNetV3_Large

Loading MobileNetV3_Large...
✓ MobileNetV3_Large loaded successfully
Probability shape: (1998, 7)
✓ MobileNetV3_Large completed and removed from GPU
Current GPU allocated memory: 32.00 MB

PROCESSING: Swin_Tiny

Loading Swin_Tiny...
✓ Swin_Tiny loaded successfully
Probability shape: (1998, 7)
✓ Swin_Tiny completed and removed from GPU
Current GPU allocated memory: 32.00 MB


In [9]:
# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_predictions(
    y_true,
    y_pred
):
    
    return {
        
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        
        "Balanced Accuracy": balanced_accuracy_score(
            y_true,
            y_pred
        ),
        
        "Precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        
        "Recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        
        "F1-Score": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    }

In [10]:
# ============================================================
# VERIFY INDIVIDUAL MODELS
# ============================================================

individual_results = []

print("=" * 80)
print("INDIVIDUAL MODEL RESULTS")
print("=" * 80)


for model_name, probabilities in all_probabilities.items():
    
    predictions = np.argmax(
        probabilities,
        axis=1
    )
    
    metrics = evaluate_predictions(
        true_labels,
        predictions
    )
    
    result = {
        "Combination": model_name,
        "Type": "Individual",
        "Weights": "-",
        **metrics
    }
    
    individual_results.append(result)
    
    
    print(f"\n{model_name}")
    
    print(
        f"Accuracy: {metrics['Accuracy'] * 100:.2f}%"
    )
    
    print(
        f"Balanced Accuracy: "
        f"{metrics['Balanced Accuracy'] * 100:.2f}%"
    )
    
    print(
        f"Macro F1: "
        f"{metrics['F1-Score'] * 100:.2f}%"
    )

INDIVIDUAL MODEL RESULTS

DenseNet121
Accuracy: 84.53%
Balanced Accuracy: 76.59%
Macro F1: 75.57%

ResNet50
Accuracy: 84.73%
Balanced Accuracy: 78.30%
Macro F1: 77.10%

EfficientNet_B0
Accuracy: 82.33%
Balanced Accuracy: 74.57%
Macro F1: 72.27%

MobileNetV3_Large
Accuracy: 83.08%
Balanced Accuracy: 75.11%
Macro F1: 74.20%

Swin_Tiny
Accuracy: 85.64%
Balanced Accuracy: 77.66%
Macro F1: 78.71%


In [11]:
# ============================================================
# SOFT VOTING FUNCTION
# ============================================================

def soft_vote(
    model_names,
    weights=None
):
    
    # Equal weights by default
    
    if weights is None:
        
        weights = [
            1 / len(model_names)
        ] * len(model_names)
    
    
    weights = np.array(
        weights,
        dtype=np.float32
    )
    
    
    # Normalize weights
    
    weights = weights / weights.sum()
    
    
    # Create empty probability array
    
    combined_probabilities = np.zeros_like(
        all_probabilities[model_names[0]]
    )
    
    
    # Combine probabilities
    
    for model_name, weight in zip(
        model_names,
        weights
    ):
        
        combined_probabilities += (
            weight *
            all_probabilities[model_name]
        )
    
    
    predictions = np.argmax(
        combined_probabilities,
        axis=1
    )
    
    
    return predictions, weights

In [12]:
# ============================================================
# TEST ALL 2-MODEL COMBINATIONS
# ============================================================

model_names = list(
    all_probabilities.keys()
)


two_model_combinations = list(
    itertools.combinations(
        model_names,
        2
    )
)


two_model_results = []

print("=" * 80)
print("TESTING ALL 2-MODEL COMBINATIONS")
print("=" * 80)

print("Total combinations:", len(two_model_combinations))


for combination in two_model_combinations:
    
    predictions, weights = soft_vote(
        model_names=combination,
        weights=[0.5, 0.5]
    )
    
    
    metrics = evaluate_predictions(
        true_labels,
        predictions
    )
    
    
    result = {
        
        "Combination": " + ".join(combination),
        
        "Type": "2-Model Equal Voting",
        
        "Weights":
            f"{weights[0]:.2f} / "
            f"{weights[1]:.2f}",
        
        **metrics
    }
    
    
    two_model_results.append(result)
    
    
    print(
        f"\n{result['Combination']}"
    )
    
    print(
        f"Accuracy: "
        f"{metrics['Accuracy'] * 100:.2f}%"
    )
    
    print(
        f"Macro F1: "
        f"{metrics['F1-Score'] * 100:.2f}%"
    )

TESTING ALL 2-MODEL COMBINATIONS
Total combinations: 10

DenseNet121 + ResNet50
Accuracy: 85.89%
Macro F1: 78.80%

DenseNet121 + EfficientNet_B0
Accuracy: 85.64%
Macro F1: 77.87%

DenseNet121 + MobileNetV3_Large
Accuracy: 85.39%
Macro F1: 77.08%

DenseNet121 + Swin_Tiny
Accuracy: 86.79%
Macro F1: 80.62%

ResNet50 + EfficientNet_B0
Accuracy: 84.83%
Macro F1: 76.59%

ResNet50 + MobileNetV3_Large
Accuracy: 85.39%
Macro F1: 78.07%

ResNet50 + Swin_Tiny
Accuracy: 86.94%
Macro F1: 80.61%

EfficientNet_B0 + MobileNetV3_Large
Accuracy: 84.13%
Macro F1: 76.32%

EfficientNet_B0 + Swin_Tiny
Accuracy: 85.99%
Macro F1: 79.01%

MobileNetV3_Large + Swin_Tiny
Accuracy: 86.29%
Macro F1: 79.35%


In [13]:
# ============================================================
# WEIGHTED 2-MODEL COMBINATIONS
# ============================================================

WEIGHT_COMBINATIONS = [
    
    [0.50, 0.50],
    
    [0.60, 0.40],
    [0.40, 0.60],
    
    [0.70, 0.30],
    [0.30, 0.70],
    
    [0.80, 0.20],
    [0.20, 0.80]
]


weighted_results = []

print("=" * 80)
print("TESTING WEIGHTED 2-MODEL COMBINATIONS")
print("=" * 80)


for combination in two_model_combinations:
    
    for weights in WEIGHT_COMBINATIONS:
        
        predictions, normalized_weights = soft_vote(
            model_names=combination,
            weights=weights
        )
        
        
        metrics = evaluate_predictions(
            true_labels,
            predictions
        )
        
        
        result = {
            
            "Combination":
                " + ".join(combination),
            
            "Type":
                "Weighted Soft Voting",
            
            "Weights":
                f"{normalized_weights[0]:.2f} / "
                f"{normalized_weights[1]:.2f}",
            
            **metrics
        }
        
        
        weighted_results.append(result)


print(
    "Total weighted experiments:",
    len(weighted_results)
)

TESTING WEIGHTED 2-MODEL COMBINATIONS
Total weighted experiments: 70


In [14]:
# ============================================================
# TEST ALL 3-MODEL COMBINATIONS
# ============================================================

three_model_combinations = list(
    itertools.combinations(
        model_names,
        3
    )
)


three_model_results = []

print("=" * 80)
print("TESTING ALL 3-MODEL COMBINATIONS")
print("=" * 80)

print(
    "Total combinations:",
    len(three_model_combinations)
)


for combination in three_model_combinations:
    
    predictions, weights = soft_vote(
        model_names=combination
    )
    
    
    metrics = evaluate_predictions(
        true_labels,
        predictions
    )
    
    
    result = {
        
        "Combination":
            " + ".join(combination),
        
        "Type":
            "3-Model Equal Voting",
        
        "Weights":
            " / ".join(
                [
                    f"{w:.2f}"
                    for w in weights
                ]
            ),
        
        **metrics
    }
    
    
    three_model_results.append(result)


print("✓ 3-model combinations completed")

TESTING ALL 3-MODEL COMBINATIONS
Total combinations: 10
✓ 3-model combinations completed


In [15]:
# ============================================================
# ALL 5 MODELS ENSEMBLE
# ============================================================

predictions, weights = soft_vote(
    model_names=model_names
)


metrics = evaluate_predictions(
    true_labels,
    predictions
)


five_model_result = {
    
    "Combination":
        " + ".join(model_names),
    
    "Type":
        "5-Model Equal Voting",
    
    "Weights":
        " / ".join(
            [
                f"{w:.2f}"
                for w in weights
            ]
        ),
    
    **metrics
}


print("=" * 80)
print("ALL 5 MODELS ENSEMBLE")
print("=" * 80)

print(
    f"Accuracy: "
    f"{metrics['Accuracy'] * 100:.2f}%"
)

print(
    f"Balanced Accuracy: "
    f"{metrics['Balanced Accuracy'] * 100:.2f}%"
)

print(
    f"Macro F1: "
    f"{metrics['F1-Score'] * 100:.2f}%"
)

ALL 5 MODELS ENSEMBLE
Accuracy: 87.04%
Balanced Accuracy: 81.55%
Macro F1: 81.33%


In [16]:
# ============================================================
# COMBINE ALL RESULTS
# ============================================================

all_results = (
    
    individual_results
    
    + two_model_results
    
    + weighted_results
    
    + three_model_results
    
    + [five_model_result]
)


results_df = pd.DataFrame(
    all_results
)


# ============================================================
# CONVERT METRICS TO PERCENTAGE
# ============================================================

metric_columns = [
    
    "Accuracy",
    
    "Balanced Accuracy",
    
    "Precision",
    
    "Recall",
    
    "F1-Score"
]


for column in metric_columns:
    
    results_df[column] = (
        results_df[column] * 100
    ).round(2)


# ============================================================
# SORT BY MACRO F1
# ============================================================

results_df = results_df.sort_values(
    by="F1-Score",
    ascending=False
).reset_index(
    drop=True
)


print("=" * 100)
print("FINAL HYBRID MODEL RANKING")
print("=" * 100)


print(
    results_df[
        [
            "Combination",
            "Type",
            "Weights",
            "Accuracy",
            "Balanced Accuracy",
            "Precision",
            "Recall",
            "F1-Score"
        ]
    ].head(20)
)

FINAL HYBRID MODEL RANKING
                                          Combination                  Type  \
0   DenseNet121 + ResNet50 + EfficientNet_B0 + Mob...  5-Model Equal Voting   
1         DenseNet121 + MobileNetV3_Large + Swin_Tiny  3-Model Equal Voting   
2                             DenseNet121 + Swin_Tiny  2-Model Equal Voting   
3                             DenseNet121 + Swin_Tiny  Weighted Soft Voting   
4                                ResNet50 + Swin_Tiny  Weighted Soft Voting   
5                                ResNet50 + Swin_Tiny  2-Model Equal Voting   
6                  DenseNet121 + ResNet50 + Swin_Tiny  3-Model Equal Voting   
7          DenseNet121 + ResNet50 + MobileNetV3_Large  3-Model Equal Voting   
8            ResNet50 + MobileNetV3_Large + Swin_Tiny  3-Model Equal Voting   
9              ResNet50 + EfficientNet_B0 + Swin_Tiny  3-Model Equal Voting   
10     ResNet50 + EfficientNet_B0 + MobileNetV3_Large  3-Model Equal Voting   
11                       

In [17]:
# ============================================================
# SAVE COMPLETE RESULTS
# ============================================================

results_path = (
    RESULTS_DIR /
    "hybrid_model_combinations.csv"
)


results_df.to_csv(
    results_path,
    index=False
)


print("=" * 80)
print("RESULTS SAVED SUCCESSFULLY")
print("=" * 80)

print(results_path)

RESULTS SAVED SUCCESSFULLY
../results/hybrid_model_combinations.csv


In [18]:
# ============================================================
# BEST HYBRID MODEL
# ============================================================

best_result = results_df.iloc[0]


print("\n" + "=" * 80)
print("BEST HYBRID MODEL")
print("=" * 80)

print(
    "Combination:",
    best_result["Combination"]
)

print(
    "Type:",
    best_result["Type"]
)

print(
    "Weights:",
    best_result["Weights"]
)

print()

print(
    f"Accuracy: "
    f"{best_result['Accuracy']:.2f}%"
)

print(
    f"Balanced Accuracy: "
    f"{best_result['Balanced Accuracy']:.2f}%"
)

print(
    f"Precision: "
    f"{best_result['Precision']:.2f}%"
)

print(
    f"Recall: "
    f"{best_result['Recall']:.2f}%"
)

print(
    f"Macro F1: "
    f"{best_result['F1-Score']:.2f}%"
)


print("\n" + "=" * 80)
print("PHASE 22 COMPLETED SUCCESSFULLY")
print("=" * 80)


BEST HYBRID MODEL
Combination: DenseNet121 + ResNet50 + EfficientNet_B0 + MobileNetV3_Large + Swin_Tiny
Type: 5-Model Equal Voting
Weights: 0.20 / 0.20 / 0.20 / 0.20 / 0.20

Accuracy: 87.04%
Balanced Accuracy: 81.55%
Precision: 81.61%
Recall: 81.55%
Macro F1: 81.33%

PHASE 22 COMPLETED SUCCESSFULLY
